# MiniAutoGen E2E Demo: AI Dev Team Builds a Terminal Tamagotchi

**4 AI agents** collaborate to develop a working Tamagotchi game — from design to code to testing to approval.

### What This Demonstrates

| Feature | How It Appears |
|---------|---------------|
| **Workflow Mode** | Architect → Developer → Tester (sequential pipeline) |
| **Deliberation Mode** | All 3 review code, Architect consolidates |
| **Outer Loop + Quality Gate** | Tech Lead approves or sends back for iteration |
| **Event System** | Full observability — who did what, when, how long |
| **Policies** | Timeout per agent, max iterations safety valve |
| **Backend Abstraction** | Gemini CLI as engine (swappable for OpenAI/Anthropic) |

### Prerequisites
- `gemini` CLI installed and authenticated
- MiniAutoGen installed (`pip install -e .`)

In [ ]:
# Cell 1: Setup — imports, helpers, pre-flight checks
import json, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import display, Markdown, HTML

# MiniAutoGen SDK — Event System
from miniautogen.core.contracts.events import ExecutionEvent
from miniautogen.core.events import EventType, InMemoryEventSink

# --- Configuration ---
WORKSPACE = Path("workspace").resolve()
MAX_ITERATIONS = 3
AGENT_TIMEOUT = 300  # 5 min per agent (code generation can be slow)

# --- Pre-flight ---
assert shutil.which("gemini"), "ERROR: 'gemini' CLI not found. Install and authenticate first."
WORKSPACE.mkdir(exist_ok=True)

# --- Event sink (captures all events for observability) ---
sink = InMemoryEventSink()
run_id = f"tamagotchi-{int(time.time())}"

# --- Helpers ---
async def emit(event_type, **payload):
    """Emit a typed event to the MiniAutoGen event sink."""
    await sink.publish(ExecutionEvent(type=event_type, run_id=run_id, payload=payload))

def call_gemini(prompt: str) -> str:
    """Call Gemini CLI via stdin (handles long prompts)."""
    result = subprocess.run(
        ["gemini"],
        input=prompt,
        capture_output=True, text=True,
        cwd=str(WORKSPACE), timeout=AGENT_TIMEOUT,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Gemini error: {result.stderr.strip()[:300]}")
    return result.stdout.strip()

def read_file(name: str) -> str:
    """Read a workspace file, empty string if missing."""
    p = WORKSPACE / name
    return p.read_text() if p.exists() else ""

print(f"Workspace: {WORKSPACE}")
print(f"Event sink ready ({EventType.__members__.__len__()} event types available)")
print(f"Run ID: {run_id}")
print("Setup complete.")

---

## Phase 1: Build (Workflow Mode)

**Coordination Mode**: `WorkflowRuntime` — sequential pipeline where each agent's output feeds the next.

```
Architect → Developer → Tester
(design)    (code)       (test)
```

The Architect designs the game structure. The Developer implements it. The Tester writes and runs tests. Each agent receives the previous agent's output as context.

In [ ]:
# Phase 1, Step 1: ARCHITECT — Design the Tamagotchi
await emit(EventType.RUN_STARTED, phase="build", iteration=1)
await emit(EventType.COMPONENT_STARTED, agent="architect")

print("ARCHITECT: Designing game structure...")
architect_output = call_gemini(
    "You are a software architect designing a terminal Tamagotchi game in Python.\n\n"
    "Design a simple terminal Tamagotchi game with these features:\n"
    "- A pet with stats: hunger, happiness, energy (0-100, start at 50)\n"
    "- Actions: feed (+20 hunger), play (+20 happiness, -10 energy), sleep (+30 energy, -10 hunger)\n"
    "- Stats decay by 5 each turn\n"
    "- Game over when any stat hits 0\n"
    "- ASCII art for pet states (happy, hungry, tired, dead)\n"
    "- Turn-based loop: show status -> player chooses action -> update stats -> repeat\n\n"
    "Output a clear design document with:\n"
    "1. Module structure (single file tamagotchi.py)\n"
    "2. Classes/functions needed\n"
    "3. Game loop pseudocode\n"
    "4. ASCII art specs"
)

await emit(EventType.AGENT_REPLIED, agent="architect", content_length=len(architect_output))
await emit(EventType.COMPONENT_FINISHED, agent="architect")

display(Markdown(f"### Architect's Design\n\n{architect_output[:2000]}"))

In [ ]:
# Phase 1, Step 2: DEVELOPER — Implement the code
await emit(EventType.COMPONENT_STARTED, agent="developer")

print("DEVELOPER: Implementing Tamagotchi code...")
developer_output = call_gemini(
    "You are a Python developer. Implement the Tamagotchi game based on the architect's design.\n\n"
    f"Design spec:\n{architect_output}\n\n"
    "Write the complete tamagotchi.py file to the current directory. The file must:\n"
    "- Be runnable with `python tamagotchi.py`\n"
    "- Include all game logic in a single file\n"
    "- Handle invalid input gracefully\n"
    "- Exit cleanly on Ctrl+C\n\n"
    "Write the file now."
)

await emit(EventType.AGENT_REPLIED, agent="developer", content_length=len(developer_output))
await emit(EventType.COMPONENT_FINISHED, agent="developer")

game_code = read_file("tamagotchi.py")
print(f"tamagotchi.py written: {len(game_code)} chars")
display(Markdown(f"### Developer's Output\n\n```python\n{game_code[:1500]}\n```\n\n*({len(game_code)} chars total)*"))

In [ ]:
# Phase 1, Step 3: TESTER — Write and run tests
await emit(EventType.COMPONENT_STARTED, agent="tester")

print("TESTER: Writing and running tests...")
game_code = read_file("tamagotchi.py")
tester_output = call_gemini(
    "You are a QA engineer. Test the Tamagotchi game that was just implemented.\n\n"
    f"Current code:\n```python\n{game_code}\n```\n\n"
    "1. Write test_tamagotchi.py to the current directory with unit tests covering:\n"
    "   - Pet initialization (stats start at 50)\n"
    "   - Feed action increases hunger\n"
    "   - Play action increases happiness, decreases energy\n"
    "   - Sleep action increases energy, decreases hunger\n"
    "   - Stats decay each turn\n"
    "   - Game over when stat hits 0\n\n"
    "2. Run the tests: python test_tamagotchi.py\n\n"
    "3. Report results: which tests pass, which fail, any errors."
)

await emit(EventType.AGENT_REPLIED, agent="tester", content_length=len(tester_output))
await emit(EventType.COMPONENT_FINISHED, agent="tester")
await emit(EventType.RUN_FINISHED, phase="build", iteration=1)

test_code = read_file("test_tamagotchi.py")
print(f"test_tamagotchi.py written: {len(test_code)} chars")
display(Markdown(f"### Tester's Report\n\n{tester_output[:1500]}"))

---

## Phase 2: Code Review (Deliberation Mode)

**Coordination Mode**: `DeliberationRuntime` — all 3 agents review the code from their own perspective, then the Architect consolidates into a unified report.

```
Architect ──┐
Developer ──┼── reviews ──→ Architect consolidates ──→ Review Report  
Tester ─────┘
```

This is the power of deliberation: **multiple perspectives** converge into a single, prioritized assessment.

In [ ]:
# Phase 2: DELIBERATION — 3 agents review, Architect consolidates
await emit(EventType.DELIBERATION_STARTED, phase="review", iteration=1)

game_code = read_file("tamagotchi.py")
test_code = read_file("test_tamagotchi.py")
reviews = {}

for agent_name, perspective in [
    ("Architect", "architecture, module structure, and design clarity"),
    ("Developer", "code quality, readability, and implementation correctness"),
    ("Tester", "test coverage, edge cases, and reliability"),
]:
    print(f"  [{agent_name.upper()}] Reviewing from {perspective} perspective...")
    await emit(EventType.COMPONENT_STARTED, agent=agent_name.lower(), sub_phase="review")
    
    review = call_gemini(
        f"You are a {agent_name} reviewing a Tamagotchi game implementation.\n\n"
        f"Review from the perspective of {perspective}.\n\n"
        f"Code:\n```python\n{game_code}\n```\n\n"
        f"Tests:\n```python\n{test_code}\n```\n\n"
        "Provide a short review (max 200 words) with:\n"
        "- What works well\n- Issues found\n- Specific improvements needed"
    )
    reviews[agent_name] = review
    await emit(EventType.AGENT_REPLIED, agent=agent_name.lower(), content_length=len(review))
    await emit(EventType.COMPONENT_FINISHED, agent=agent_name.lower(), sub_phase="review")
    print(f"  Review submitted ({len(review)} chars)")

await emit(EventType.DELIBERATION_ROUND_COMPLETED, round=1, participants=list(reviews.keys()))

# Architect consolidates
print("\n  [ARCHITECT] Consolidating all reviews...")
all_reviews = "\n\n".join(f"--- {n} Review ---\n{t}" for n, t in reviews.items())
review_report = call_gemini(
    "You are the lead Architect. Consolidate these three reviews into a single, concise report.\n\n"
    f"{all_reviews}\n\n"
    "Output a consolidated review report (max 300 words) that:\n"
    "- Summarizes consensus points\n- Lists all issues found\n"
    "- Prioritizes what to fix (critical vs nice-to-have)"
)
await emit(EventType.DELIBERATION_FINISHED, report_length=len(review_report))

display(Markdown(f"### Consolidated Review Report\n\n{review_report}"))

---

## Phase 3: Tech Lead Decision (Quality Gate)

**Coordination Pattern**: The Tech Lead acts as a **quality gate** — receives the review report, evaluates the code, and decides:
- **APPROVED** → development is done, game is ready
- **NEEDS_WORK** → specific feedback, team iterates

This is the outer loop that drives the whole process. In a full MiniAutoGen deployment, this maps to the `AgenticLoopRuntime` with the Tech Lead as the router agent.

In [ ]:
# Phase 3: TECH LEAD — Quality gate decision
await emit(EventType.COMPONENT_STARTED, agent="tech_lead")

game_code = read_file("tamagotchi.py")
test_code = read_file("test_tamagotchi.py")

print("TECH LEAD: Evaluating the team's work...")
tl_output = call_gemini(
    "You are a Tech Lead evaluating the team's work on a Tamagotchi game.\n\n"
    f"Review report:\n{review_report}\n\n"
    f"Current code:\n```python\n{game_code}\n```\n\n"
    f"Current tests:\n```python\n{test_code}\n```\n\n"
    'Evaluate the work. Respond with ONLY a JSON object:\n'
    '{\n'
    '  "verdict": "APPROVED" or "NEEDS_WORK",\n'
    '  "feedback": "specific actionable feedback if NEEDS_WORK, or congratulations if APPROVED"\n'
    '}\n\n'
    "Criteria for APPROVED:\n"
    "- Code runs without errors\n"
    "- At least 4/6 core tests pass\n"
    "- Game loop works (feed, play, sleep actions functional)\n"
    "- Code is clean and readable\n\n"
    "If ANY test crashes or the game doesn't start, verdict MUST be NEEDS_WORK."
)

await emit(EventType.AGENT_REPLIED, agent="tech_lead", content_length=len(tl_output))
await emit(EventType.COMPONENT_FINISHED, agent="tech_lead")

# Parse verdict
verdict, feedback = "NEEDS_WORK", ""
try:
    raw = tl_output
    if "```" in raw:
        for part in raw.split("```"):
            s = part.strip()
            if s.startswith("json"): s = s[4:].strip()
            if s.startswith("{"): raw = s; break
    parsed = json.loads(raw)
    verdict = parsed.get("verdict", "NEEDS_WORK")
    feedback = parsed.get("feedback", "")
except (json.JSONDecodeError, AttributeError):
    feedback = f"Could not parse response: {tl_output[:200]}"

emoji = "✅" if verdict == "APPROVED" else "🔄"
display(Markdown(f"### Tech Lead Verdict: {emoji} {verdict}\n\n{feedback}"))

---

## Event System — Full Observability

One of MiniAutoGen's key differentiators: **every action emits typed events** to composable sinks. No coupling between the coordination logic and the observability layer.

Let's inspect the event stream from the pipeline we just ran.

In [ ]:
# Event System: Inspect the captured event stream
print(f"Total events captured: {len(sink.events)}\n")

# Event breakdown by type
event_counts = {}
for ev in sink.events:
    event_counts[ev.type] = event_counts.get(ev.type, 0) + 1

print("Event breakdown:")
print("-" * 40)
for etype, count in sorted(event_counts.items()):
    print(f"  {etype:40s} {count}")

# Timeline view
print(f"\nTimeline (first 15 events):")
print("-" * 60)
for i, ev in enumerate(sink.events[:15]):
    payload = ev.payload_dict() if hasattr(ev, 'payload_dict') else dict(ev.payload) if ev.payload else {}
    agent = payload.get("agent", "system")
    phase = payload.get("phase", payload.get("sub_phase", ""))
    print(f"  {i+1:2d}. {ev.type:35s} agent={agent:12s} {phase}")

---

## Play Your Tamagotchi!

If the Tech Lead approved the work, the game should be ready to play.

> **Note**: The cell below runs the game interactively in a subprocess. You can also run it directly in a terminal: `python workspace/tamagotchi.py`

In [ ]:
# Check the final result
game_path = WORKSPACE / "tamagotchi.py"

if game_path.exists():
    print(f"Game file: {game_path} ({game_path.stat().st_size} bytes)")
    print(f"\nTo play, run in your terminal:")
    print(f"  python {game_path}")
    print(f"\nFirst 30 lines of the game:")
    print("-" * 50)
    print("\n".join(game_path.read_text().splitlines()[:30]))
else:
    print("Game file not found — agents may not have written it successfully.")